In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Embedding modelimizi yükleyelim
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Bilgi tabanımız (Chunk'larımız - İndeks numaraları 0, 1, 2, 3)
chunks = [
    "Python bir programlama dilidir.",             # Chunk 0
    "Overfitting modelin ezberlemesidir.",         # Chunk 1
    "İstanbul kalabalık bir şehirdir.",           # Chunk 2
    "Random Forest ağaçların oyunu birleştirir."   # Chunk 3
]

# 3. Chunk'ları vektörleştirip FAISS indeksine ekleyelim
embeddings = model.encode(chunks).astype('float32')
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(384)
index.add(embeddings)

# 4. ALTIN STANDART (TEST SETİ): (Soru, Doğru Chunk İndeksi)
# Neden yapıyoruz? Sistemin doğru chunk'ı bulup bulamadığını kıyaslayacağımız 'Gerçek Cevap Anahtarı' bu.
test_seti = [
    ("Model neden ezber yapar?", 1),       # Cevap Chunk 1'de olmalı
    ("Overfitting nedir?", 1),             # Cevap Chunk 1'de olmalı
    ("Python bir dil midir?", 0),          # Cevap Chunk 0'da olmalı
    ("Yazılım dili olarak ne kullanılır?", 0), # Cevap Chunk 0'da olmalı
    ("Ağaçlar nasıl birleştirilir?", 3),   # Cevap Chunk 3'te olmalı
    ("Ensemble öğrenme ve ağaçlar?", 3),   # Cevap Chunk 3'te olmalı
    ("Türkiye'nin en kalabalık şehri?", 2), # Cevap Chunk 2'de olmalı
    ("İstanbul nasıl bir yerdir?", 2)      # Cevap Chunk 2'de olmalı
]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\elif\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\elif\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
def hit_rate_hesapla(test_seti, k=1):
    isabet_sayisi = 0
    
    for soru, dogru_chunk_idx in test_seti:
        # Soruyu vektör yapıp normalize ediyoruz
        q = model.encode([soru]).astype('float32')
        faiss.normalize_L2(q)
        
        # FAISS'ten en yakın 'k' adet sonucu istiyoruz
        _, getirilen_indeksler = index.search(q, k)
        
        # Doğru chunk, FAISS'in getirdiği listede var mı?
        if dogru_chunk_idx in getirilen_indeksler[0]:
            isabet_sayisi += 1
            
    # Toplam isabet oranını dönüyoruz (Örn: 8 soruda 8 isabet = 1.0 yani %100)
    return isabet_sayisi / len(test_seti)

# Farklı k değerlerini test edelim
print("=== RETRIEVAL BAŞARI METRİKLERİ (HIT-RATE) ===")
print(f"Hit-Rate@1: %{hit_rate_hesapla(test_seti, k=1) * 100:.1f}")
print(f"Hit-Rate@3: %{hit_rate_hesapla(test_seti, k=3) * 100:.1f}")
print(f"Hit-Rate@5: %{hit_rate_hesapla(test_seti, k=5) * 100:.1f}")

=== RETRIEVAL BAŞARI METRİKLERİ (HIT-RATE) ===
Hit-Rate@1: %75.0
Hit-Rate@3: %100.0
Hit-Rate@5: %100.0


In [11]:
def baglilik_kontrolu(cevap, baglam):
    # Modele ne yapacağını örneklerle gösteriyoruz (Few-shot prompting)
    prompt = f"""[CONTEXT]: {baglam}
[ANSWER]: {cevap}

Is the answer fully supported by the context? Answer strictly with YES or NO.
[VERDICT]:"""

    cikti = pipe(prompt, max_new_tokens=3, do_sample=False)[0]['generated_text']
    
    # VERDICT etiketinden sonrasını alıyoruz
    hakem_yaniti = cikti.split("[VERDICT:]")[-1].strip().upper()
    
    # Türkçe karşılığına çevirelim
    if "YES" in hakem_yaniti:
        return "EVET"
    elif "NO" in hakem_yaniti:
        return "HAYIR"
    else:
        return f"BELİRSİZ ({hakem_yaniti})"

# --- TEST 1: Sadık / Doğru Senaryo ---
baglam_1 = "Overfitting modelin ezberlemesidir."
cevap_1 = "Overfitting, modelin ezber yapması durumudur."

# --- TEST 2: Sadık Olmayan / Uydurma Senaryo (Bozan durum) ---
baglam_2 = "Overfitting modelin ezberlemesidir."
cevap_2 = "Overfitting modelin ezber yapmasıdır ve genellikle yapay sinir ağlarında 500. epoch'tan sonra görülür."

print("--- FAITHFULNESS (SADAKAT) TESTİ ---")
print(f"Test 1 (Doğru Bağlam) Hakem Kararı : {baglilik_kontrolu(cevap_1, baglam_1)}")
print(f"Test 2 (Bozulmuş/Uydurma) Hakem Kararı: {baglilik_kontrolu(cevap_2, baglam_2)}")

[transformers] Both `max_new_tokens` (=3) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- FAITHFULNESS (SADAKAT) TESTİ ---


[transformers] Both `max_new_tokens` (=3) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test 1 (Doğru Bağlam) Hakem Kararı : EVET
Test 2 (Bozulmuş/Uydurma) Hakem Kararı: EVET


In [13]:
# 1. ARAMA (Retrieval): FAISS'e sorumuzu sorup ilgili bağlamı buluyoruz
soru = "Overfitting nedir?"

q = model.encode([soru]).astype('float32')
faiss.normalize_L2(q)
_, indeksler = index.search(q, 1) # En başta kurduğumuz 4 chunk'lı indeksimizi kullanıyoruz

# Bulunan bağlam (Chunk 1 - Overfitting cümlesi gelecek)
bulunan_baglam = chunks[indeksler[0][0]]

# 2. ÜRETİM (Generation): Modelden gelen bağlama göre cevap üretmesini istiyoruz
prompt = f"""Aşağıdaki bağlamı kullanarak soruyu yanıtla.
BAĞLAM: {bulunan_baglam}
SORU: {soru}
CEVAP:"""

ham_cikti = pipe(prompt, max_new_tokens=20, do_sample=False)[0]['generated_text']
uretilen_cevap = ham_cikti.split("CEVAP:")[-1].strip()

# 3. ÖLÇÜM (Faithfulness): Hakem fonksiyonumuzla sadakati ölçüyoruz
hakem_karari = baglilik_kontrolu(uretilen_cevap, bulunan_baglam)

print("=== UÇTAN UCA RAG VE SADAKAT ÖLÇÜMÜ ===")
print(f"Soru            : {soru}")
print(f"Bulunan Bağlam  : {bulunan_baglam}")
print(f"Üretilen Cevap  : {uretilen_cevap}")
print(f"Hakem Kararı    : {hakem_karari}")

[transformers] Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=3) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== UÇTAN UCA RAG VE SADAKAT ÖLÇÜMÜ ===
Soru            : Overfitting nedir?
Bulunan Bağlam  : Overfitting modelin ezberlemesidir.
Üretilen Cevap  : Overfitting, modelinizi daha fazla öğrenmeyi sağlamak
Hakem Kararı    : EVET


In [14]:
# 1. BİLEREK ALAKASIZ / BOZUK BİR SORU SORUYORUZ
bozuk_soru = "Karar ağaçlarında Gini katsayısı nasıl hesaplanır?"

# Soruyu FAISS'e soruyoruz (Veritabanımızda Gini katsayısı ile ilgili HİÇBİR BİLGİ YOK!)
q = model.encode([bozuk_soru]).astype('float32')
faiss.normalize_L2(q)
_, indeksler = index.search(q, 1)

# FAISS mecburen en az uzak olan parçayı getirecek (Örn: Random Forest veya Overfitting cümlesi)
getirilen_alakasiz_baglam = chunks[indeksler[0][0]]

# 2. GENERATION: Modelden bu alakasız bağlamla cevabı üretmesini istiyoruz
prompt = f"""Aşağıdaki bağlamı kullanarak soruyu yanıtla.
BAĞLAM: {getirilen_alakasiz_baglam}
SORU: {bozuk_soru}
CEVAP:"""

ham_cikti = pipe(prompt, max_new_tokens=25, do_sample=False)[0]['generated_text']
uretilen_cevap = ham_cikti.split("CEVAP:")[-1].strip()

# 3. FAITHFULNESS (SADAKAT) KONTROLÜ
hakem_karari = baglilik_kontrolu(uretilen_cevap, getirilen_alakasiz_baglam)

print("=== BOZUK / BAĞLAM DIŞI SORU TESTİ ===")
print(f"Bozuk Soru       : {bozuk_soru}")
print(f"Getirilen Bağlam : {getirilen_alakasiz_baglam}")
print(f"Üretilen Cevap   : {uretilen_cevap}")
print(f"Hakem Kararı     : {hakem_karari}")

[transformers] Both `max_new_tokens` (=25) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=3) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== BOZUK / BAĞLAM DIŞI SORU TESTİ ===
Bozuk Soru       : Karar ağaçlarında Gini katsayısı nasıl hesaplanır?
Getirilen Bağlam : İstanbul kalabalık bir şehirdir.
Üretilen Cevap   : Gini katsayısının nasıl hesaplanacağını söyle
Hakem Kararı     : EVET


In [15]:
# 1. GERÇEK BELGENİ OKUMA
with open("staj_notlari.txt", "r", encoding="utf-8") as f:
    gercek_belge = f.read()

# STRATEJİ A: Sabit Kelime Penceresi + Overlap (Örtüşme)
def kelime_chunk_olustur(metin, boyut=20, overlap=5):
    kelimeler = metin.split()
    parcalar = []
    i = 0
    while i < len(kelimeler):
        parca = " ".join(kelimeler[i:i + boyut])
        parcalar.append(parca)
        i += boyut - overlap  # Overlap kadar geri geliyoruz
    return parcalar

# STRATEJİ B: Paragraf Bazlı Bölme (Doğal Sınırlar)
def paragraf_chunk_olustur(metin):
    return [p.strip() for p in metin.split("\n\n") if p.strip()]

# Her iki stratejiyi kendi belgen üzerinde çalıştıralım
chunks_A = kelime_chunk_olustur(gercek_belge, boyut=20, overlap=5)
chunks_B = paragraf_chunk_olustur(gercek_belge)

print(f"Strateji A (Kelime + Overlap) Toplam Chunk Sayısı : {len(chunks_A)}")
print(f"Strateji B (Paragraf Bazlı) Toplam Chunk Sayısı   : {len(chunks_B)}")

Strateji A (Kelime + Overlap) Toplam Chunk Sayısı : 6
Strateji B (Paragraf Bazlı) Toplam Chunk Sayısı   : 3


In [16]:
# Strateji A'daki ilk iki chunk'ı yazdıralım
print("=== CHUNK 0 ===")
print(chunks_A[0])

print("\n=== CHUNK 1 ===")
print(chunks_A[1])

=== CHUNK 0 ===
Yapay zeka sistemlerinde overfitting modelin veriyi ezberlemesidir. Model eğitim verisini ezberlediğinde gerçek hayattaki yeni verilerde başarısız olur. Bunu önlemek için

=== CHUNK 1 ===
başarısız olur. Bunu önlemek için düzenlileştirme ve erken durdurma teknikleri kullanılır. Random forest algoritması birden fazla karar ağacını birleştirerek kolektif


In [17]:
# 1. STRATEJİ A (Kelime + Overlap) İÇİN FAISS İNDEKSİ
embeddings_A = model.encode(chunks_A).astype('float32')
faiss.normalize_L2(embeddings_A)

index_A = faiss.IndexFlatIP(384)
index_A.add(embeddings_A)

# 2. STRATEJİ B (Paragraf Bazlı) İÇİN FAISS İNDEKSİ
embeddings_B = model.encode(chunks_B).astype('float32')
faiss.normalize_L2(embeddings_B)

index_B = faiss.IndexFlatIP(384)
index_B.add(embeddings_B)

# -------------------------------------------------------------
# TEST SETİ (Soru ve Belgedeki Doğru Cevap Konumu)
# Paragraf 0: Overfitting | Paragraf 1: Random Forest | Paragraf 2: FAISS/IVFFlat
test_sorulari = [
    ("Overfitting durumunda ne yapılır?", "önlemek için düzenlileştirme"),
    ("Random forest ezber riskini nasıl azaltır?", "birden fazla karar ağacını"),
    ("IVFFlat indeksi veriyi nasıl böler?", "Voronoi hücrelerine")
]

# Hit-Rate Ölçüm Fonksiyonu (Metin İçi Doğruluk Kontrolü)
def hit_rate_olculer(indeks, chunk_listesi, test_seti, k=1):
    isabet = 0
    for soru, aranan_metin in test_seti:
        q = model.encode([soru]).astype('float32')
        faiss.normalize_L2(q)
        _, getirilen_idxler = indeks.search(q, k)
        
        # Getirilen k adet chunk içinde aranan anahtar kelime var mı?
        bulundu = False
        for idx in getirilen_idxler[0]:
            if aranan_metin in chunk_listesi[idx]:
                bulundu = True
                break
        if bulundu:
            isabet += 1
            
    return isabet / len(test_seti)

# İki stratejiyi karşılaştıralım
hr_A = hit_rate_olculer(index_A, chunks_A, test_sorulari, k=1)
hr_B = hit_rate_olculer(index_B, chunks_B, test_sorulari, k=1)

print("=== CHUNKING STRATEJİLERİ HİT-RATE@1 KARŞILAŞTIRMASI ===")
print(f"Strateji A (Kelime + Overlap) Hit-Rate@1 : %{hr_A * 100:.1f}")
print(f"Strateji B (Paragraf Bazlı)  Hit-Rate@1 : %{hr_B * 100:.1f}")

=== CHUNKING STRATEJİLERİ HİT-RATE@1 KARŞILAŞTIRMASI ===
Strateji A (Kelime + Overlap) Hit-Rate@1 : %66.7
Strateji B (Paragraf Bazlı)  Hit-Rate@1 : %100.0


In [18]:
# --- 1. SEÇTİĞİMİZ EN İYİ CHUNKING İLE FAISS İNDEKSİNİ KURUYORUZ ---
# Paragraf bazlı (Strateji B) %100 Hit-Rate verdiği için bunu seçiyoruz
chunks = paragraf_chunk_olustur(gercek_belge)
V = model.encode(chunks).astype('float32')
faiss.normalize_L2(V)

index_son = faiss.IndexFlatIP(V.shape[1])
index_son.add(V)

# --- 2. RAG SORU-CEVAP FONKSİYONUMUZ (`rag_sor`) ---
def rag_sor(soru, aktif_index, aktif_chunks):
    # Arama (Retrieval)
    q = model.encode([soru]).astype('float32')
    faiss.normalize_L2(q)
    _, indeksler = aktif_index.search(q, 1)
    getirilen_baglam = aktif_chunks[indeksler[0][0]]
    
    # Üretim (Generation)
    prompt = f"""Sadece aşağıdaki bağlamı kullanarak soruyu yanıtla.
BAĞLAM: {getirilen_baglam}
SORU: {soru}
CEVAP:"""
    
    ham_cikti = pipe(prompt, max_new_tokens=30, do_sample=False)[0]['generated_text']
    cevap = ham_cikti.split("CEVAP:")[-1].strip()
    
    return getirilen_baglam, cevap

# --- 3. BELGENE DAİR 3-4 SORU İLE TEST ETME ---
test_sorulari = [
    "Overfitting'i önlemek için hangi teknikler kullanılır?",
    "Random forest algoritması ezber yapma riskini nasıl azaltır?",
    "nprobe parametresi tam olarak neyi belirler?"
]

print("=== GERÇEK BELGE İLE RAG SORGULAMA TESTİ ===\n")
for i, s in enumerate(test_sorulari, 1):
    baglam, cevap = rag_sor(s, index_son, chunks)
    print(f"Soru {i} : {s}")
    print(f"Bağlam : {baglam}")
    print(f"Cevap  : {cevap}")
    print("-" * 50)

[transformers] Both `max_new_tokens` (=30) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== GERÇEK BELGE İLE RAG SORGULAMA TESTİ ===



[transformers] Both `max_new_tokens` (=30) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Soru 1 : Overfitting'i önlemek için hangi teknikler kullanılır?
Bağlam : FAISS kütüphanesi vektör arama işlemlerini hızlandırmak için Facebook tarafından geliştirilmiştir. IVFFlat indeksi veriyi Voronoi hücrelerine bölerek kümeleme yapar. nprobe parametresi ise arama sırasında kaç hücreye bakılacağını belirler.
Cevap  : Overfitting'i önlemek için kullanılan tekniklerden birini seçin.
--------------------------------------------------


[transformers] Both `max_new_tokens` (=30) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Soru 2 : Random forest algoritması ezber yapma riskini nasıl azaltır?
Bağlam : Random forest algoritması birden fazla karar ağacını birleştirerek kolektif bir karar mekanizması oluşturur. Bu yöntem tek bir karar ağacının ezber yapma riskini ciddi oranda azaltır.
Cevap  : Random forest algoritması, birden fazla karar ağacını birleştirerek kolektif bir
--------------------------------------------------
Soru 3 : nprobe parametresi tam olarak neyi belirler?
Bağlam : FAISS kütüphanesi vektör arama işlemlerini hızlandırmak için Facebook tarafından geliştirilmiştir. IVFFlat indeksi veriyi Voronoi hücrelerine bölerek kümeleme yapar. nprobe parametresi ise arama sırasında kaç hücreye bakılacağını belirler.
Cevap  : nprobe parametresi, arama sırasında kaç hücreye bakılacağını
--------------------------------------------------
